# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup
This notebook audits both the FlyRank research paper and my Week-5
future-decline model.

The model question is:

>Can historical page-level search and content signals measured before
the prediction point help identify pages that later experience a
20% decline in impressions?

Feature window:
`2026-03-01 to 2026-03-31`

Target window:
`2026-04-01 to 2026-04-30`

The main validation improvement in this notebook is a grouped split
by client. This is more conservative than a random row split because
pages from the same client may share common characteristics.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy matplotlib

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-04-01"

TARGET_START = "2026-04-01"
TARGET_END = "2026-05-01"

print("Setup complete.")

Setup complete.


In [3]:
HF_TOKEN = (os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): "))

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf " f"(TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":                  f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":                  f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":                   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d":               f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, source in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {source}").fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

Paste your Hugging Face READ token (hf_...): ··········
dim_clients                   104 rows
dim_content               519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_query_90d          2,414,248 rows


In [4]:
date_check = con.sql(
    f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS rows
    FROM {TABLES["fact_daily"]}
    """
).df()

display(date_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2025-01-27,2026-06-30,78835655


In [6]:
features = con.sql(
    f"""
    WITH feature_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_prev30,
            SUM(gsc_clicks) AS clk_prev30,
            AVG(gsc_avg_position) AS avg_position_prev30,
            COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date < DATE '{FEATURE_END}' AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_prev30,
        f.clk_prev30,
        f.avg_position_prev30,
        f.days_with_impressions,
        DATE_DIFF('day', c.content_created_date, DATE '{TARGET_START}') AS content_age_days
    FROM feature_window f
    LEFT JOIN {TABLES["dim_content"]} c ON f.client_hash_id = c.client_hash_id AND f.content_hash_id = c.content_hash_id
    WHERE f.imp_prev30 >= 100
    """
).df()

print(f"Feature rows: {len(features):,}")
display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 101,441


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,days_with_impressions,content_age_days
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331.0,2.0,14.129210,31,176
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145.0,0.0,8.470926,30,176
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,461.0,0.0,14.859827,31,176
3,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,232.0,0.0,12.297523,29,176
4,client_0797ff3a1fc9a6a5,content_27f8100281413b37,464.0,0.0,9.554793,18,176


In [8]:
future_outcomes = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_next30,
        SUM(gsc_clicks) AS clk_next30
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '{TARGET_START}' AND report_date < DATE '{TARGET_END}' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    """
).df()

print("Future outcome rows:", f"{len(future_outcomes):,}")
display(future_outcomes.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future outcome rows: 194,760


,client_hash_id,content_hash_id,imp_next30,clk_next30
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0,0.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0,0.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0,0.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0,1.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0,0.0


In [9]:
data = features.merge(future_outcomes, on=["client_hash_id", "content_hash_id"], how="inner")

print("Final modeling rows:", f"{len(data):,}")
display(data.head())

Final modeling rows: 100,893


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,days_with_impressions,content_age_days,imp_next30,clk_next30
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331.0,2.0,14.129210,31,176,561.0,3.0
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145.0,0.0,8.470926,30,176,23.0,0.0
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,461.0,0.0,14.859827,31,176,964.0,0.0
3,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,232.0,0.0,12.297523,29,176,248.0,0.0
4,client_0797ff3a1fc9a6a5,content_27f8100281413b37,464.0,0.0,9.554793,18,176,1138.0,0.0


In [10]:
data["impression_change_pct"] = (data["imp_next30"] - data["imp_prev30"]) / data["imp_prev30"]
data["is_declining"] = (data["impression_change_pct"] < -0.20).astype(int)

print(data["is_declining"].value_counts().rename({0: "not_declining", 1: "declining"}))
print("\nDecline rate:")
print(round(data["is_declining"].mean(), 4))

is_declining
declining        51944
not_declining    48949
Name: count, dtype: int64

Decline rate:
0.5148


In [12]:
FEATURE_COLS = [
    "imp_prev30",
    "clk_prev30",
    "avg_position_prev30",
    "days_with_impressions",
    "content_age_days"
]

model_data = data.dropna(subset=FEATURE_COLS + ["is_declining"]).copy()

print("Rows:", len(model_data))
print("Features:", FEATURE_COLS)

display(model_data[FEATURE_COLS + ["is_declining"]].describe().T)

Rows: 100893
Features: ['imp_prev30', 'clk_prev30', 'avg_position_prev30', 'days_with_impressions', 'content_age_days']


,count,mean,std,min,25%,50%,75%,max
imp_prev30,100893.0,2761.300952,6961.642086,100.000000,286.000000,794.000000,2532.000000,617124.000000
clk_prev30,100893.0,8.078063,34.977339,0.000000,0.000000,1.000000,6.000000,5668.000000
avg_position_prev30,100893.0,14.425139,14.231891,0.013793,4.981475,8.720766,19.189464,93.693397
days_with_impressions,100893.0,28.552377,4.959123,1.000000,29.000000,31.000000,31.000000,31.000000
content_age_days,100893.0,190.038774,127.273989,2.000000,72.000000,188.000000,266.000000,495.000000
is_declining,100893.0,0.514842,0.499782,0.000000,0.000000,1.000000,1.000000,1.000000


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Paper finding #1 — Refresh timing

The paper reports that older content refreshed within 30 days showed
a large difference in measured health and impressions compared with
older content that had not recently been refreshed.

The paper describes a **3.2x** health increase and a **57x** impression
difference in the reported comparison. It also warns that the
**361+** day bucket is unstable because it contains only one declining
page.

#### ***Methodology Question***

***Where does the outcome come from, and what is the comparison design?***

The paper's reported outcome is measured performance such as health
and impressions. However, the comparison is observational rather
than a randomized treatment experiment.

Therefore, the key methodology question is:

>Were recently refreshed pages comparable to non-refreshed pages before
the refresh, or could page age, historical visibility, content quality,
or selection into the refresh group explain part of the observed
difference?

The paper itself describes the study as observational and says that
correlations do not prove causation.

So I would treat this as an observed directional association rather
than evidence that refreshing a page necessarily caused the measured
increase.


---





### Paper finding #2 — Engagement and visibility

The paper reports that stronger engagement patterns and more consistent
visibility occur together.

For example, the paper reports a health difference between pages with
consistent visibility and pages with sporadic visibility.

#### ***Methodology question***

>What exactly is the outcome being compared, and does the validation
design support a causal interpretation?

The measured outcome is portfolio health and related search metrics.
The comparison is a grouped observational comparison, not a controlled
experiment.

Therefore, the methodology question is whether other variables such
as page age, content type, search demand, or historical visibility
could contribute to the observed difference.

The result supports an observed association in this portfolio, but
does not by itself establish that increasing engagement will cause
higher search performance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Model

I use **Logistic Regression** as the first honest model.

This is a binary classification problem with an observed future label.
Logistic Regression is appropriate because it provides a simple,
interpretable reference model before adding more complex models.

StandardScaler is used because the features have very different
numeric scales.

The client ID is never used as a feature.
It is used only for grouping the validation split.

In [13]:
def make_logistic_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])

In [18]:
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k = labels[order[:k]]

    return top_k.mean()


def evaluate_model(model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "precision@20": precision_at_k(probabilities, y_test, 20),
        "precision@50": precision_at_k(probabilities, y_test, 50),
        "precision@100": precision_at_k(probabilities, y_test, 100),
        "confusion_matrix": confusion_matrix(y_test, predictions),
    }

    return model, results, probabilities, predictions

### BEFORE — Week-5 random split

In [19]:
X = model_data[FEATURE_COLS]
y = model_data["is_declining"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)


print("Random split")
print("Train:", len(X_train_random))
print("Test :", len(X_test_random))
print("Train base rate:", round(y_train_random.mean(), 3))
print("Test base rate :", round(y_test_random.mean(), 3))

Random split
Train: 75669
Test : 25224
Train base rate: 0.515
Test base rate : 0.515


In [21]:
random_model = make_logistic_model()

random_model, random_results, random_prob, random_pred = evaluate_model(random_model, X_train_random, X_test_random, y_train_random, y_test_random)

random_results

{'accuracy': 0.6005788138281002,
 'precision': 0.5788589694966679,
 'recall': 0.8227321731095025,
 'f1': 0.6795789205864581,
 'roc_auc': np.float64(0.6498332509746186),
 'precision@20': np.float64(0.65),
 'precision@50': np.float64(0.66),
 'precision@100': np.float64(0.64),
 'confusion_matrix': array([[ 4465,  7773],
        [ 2302, 10684]])}

### AFTER — Grouped split

#### ***Honest grouped split***

The same client can have many content pages. A random split can therefore place pages from the same client in both training and testing. That makes the test set less independent. I therefore use a grouped split where all pages from a client remain in the same partition.

This asks a harder question:

>Can the model generalize to clients that were not represented in
training?

In [22]:
groups = model_data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Grouped split")
print("Train rows:", len(X_train_group))
print("Test rows :", len(X_test_group))

print("Train clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())
print("Client overlap:", len(set(groups_train).intersection(set(groups_test))))

print("Train base rate:", round(y_train_group.mean(), 3))
print("Test base rate:", round(y_test_group.mean(), 3))

Grouped split
Train rows: 84384
Test rows : 16509
Train clients: 32
Test clients: 11
Client overlap: 0
Train base rate: 0.537
Test base rate: 0.402


In [23]:
group_model = make_logistic_model()

group_model, group_results, group_prob, group_pred = evaluate_model(group_model, X_train_group, X_test_group, y_train_group, y_test_group)

group_results

{'accuracy': 0.5119631716033679,
 'precision': 0.43987849126655976,
 'recall': 0.786037394451146,
 'f1': 0.5640859167884001,
 'roc_auc': np.float64(0.620785633130692),
 'precision@20': np.float64(0.6),
 'precision@50': np.float64(0.46),
 'precision@100': np.float64(0.54),
 'confusion_matrix': array([[3239, 6638],
        [1419, 5213]])}

### Before vs after table

In [25]:
comparison = pd.DataFrame(
    [{"validation": "Week-5 random split", **random_results},
     {"validation": "Grouped by client", **group_results}])

display(comparison.round(3))

,validation,accuracy,precision,recall,f1,roc_auc,precision@20,precision@50,precision@100,confusion_matrix
0,Week-5 random split,0.601,0.579,0.823,0.680,0.650,0.65,0.66,0.64,"[[4465, 7773], [2302, 10684]]"
1,Grouped by client,0.512,0.440,0.786,0.564,0.621,0.60,0.46,0.54,"[[3239, 6638], [1419, 5213]]"


The random split is the less conservative estimate because pages from
the same client can appear in both training and testing.

The grouped result is the more relevant estimate for cross-client
generalization.

If performance decreases under the grouped split, that gap is itself
useful evidence: some of the random-split performance may depend on
client-specific structure shared across pages.

I do not interpret a grouped score as proof of deployment performance.
It is a more conservative validation estimate for this dataset.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [26]:
feature_audit = pd.DataFrame({
    "feature": FEATURE_COLS,
    "uses_future_window": [False, False, False, False, False],
    "label_derived": [False, False, False, False, False],
    "product_decision": [False, False, False, False, False]
})

display(feature_audit)

,feature,uses_future_window,label_derived,product_decision
0,imp_prev30,False,False,False
1,clk_prev30,False,False,False
2,avg_position_prev30,False,False,False
3,days_with_impressions,False,False,False
4,content_age_days,False,False,False


In [28]:
suspicious_terms = [
    "label",
    "declin",
    "future",
    "next30",
    "trend",
    "priority",
    "health_score",
    "action_type",
    "flag",
    "score"
]

suspicious_columns = [c for c in model_data.columns if any(term in c.lower() for term in suspicious_terms)]

print("Potentially suspicious columns:")
print(suspicious_columns)

Potentially suspicious columns:
['imp_next30', 'clk_next30', 'is_declining']


In [29]:
LEAKY_FEATURES = FEATURE_COLS + ["impression_change_pct"]

X_leaky = model_data[LEAKY_FEATURES]
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(X_leaky, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)


leaky_model = make_logistic_model()
leaky_model, leaky_results, leaky_prob, leaky_pred = evaluate_model(leaky_model, X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky)

print("HONEST RANDOM MODEL")
print(round(random_results["roc_auc"], 3))

print()

print("LEAKY MODEL")
print(round(leaky_results["roc_auc"], 3))

HONEST RANDOM MODEL
0.65

LEAKY MODEL
1.0



The deliberately leaky model performs dramatically better than
the honest model because `impression_change_pct` is calculated using
the future target window. This is not useful predictive performance.

It is a successful leakage test: the experiment demonstrates that the
validation pipeline can detect a feature that directly contains target
information.

The feature is therefore excluded from the final model.

In [30]:
product_fields = [
    "priority_score",
    "health_score",
    "action_type",
    "refresh_tier",
    "flag"
]

found_product_fields = [c for c in model_data.columns if c.lower() in [x.lower() for x in product_fields]]

print("Product decision fields found in model data:", found_product_fields)

assert len(found_product_fields) == 0

print("PASS: no product decision fields are used.")

Product decision fields found in model data: []
PASS: no product decision fields are used.


In [32]:
perm = permutation_importance(group_model, X_test_group, y_test_group, scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)

perm_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(perm_df.round(4))

,feature,importance_mean,importance_std
3,days_with_impressions,0.0980,0.0014
1,clk_prev30,0.0695,0.0028
2,avg_position_prev30,0.0048,0.0010
0,imp_prev30,0.0007,0.0011
4,content_age_days,-0.0029,0.0012


The coefficient ranking describes which standardized features the
logistic model uses most strongly within this fitted sample.

A positive coefficient means higher values are associated with higher
predicted probability of decline, while a negative coefficient means
the opposite direction.

These are directional model associations, not causal effects.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

"My Logistic Regression model predicts which pages will decline in
search performance."

### Safer claim

"On the evaluated March-to-April 2026 warehouse slice, the Logistic
Regression model produced measured ranking and classification
performance using five historical page-level features. Under a
client-grouped validation split, the model provides a directional
decision-support signal for prioritizing pages that may experience a
future `> 20%` impression decline. The result should not be interpreted
as proof that the model will generalize to all future clients or time
periods."

### Why I changed the claim

The model was evaluated on one historical window.

The grouped split is more conservative than the original random split,
but it still does not establish future production performance.

The dataset is observational, so the model identifies statistical
associations rather than causal effects.

### What I learned from the paper audit

The FlyRank paper explicitly separates direct portfolio evidence from
exploratory ML analysis. Its methodology states that the ML appendix
uses 80/20 splits for Logistic Regression, Random Forest, and a shallow
Decision Tree, while the headline findings primarily rely on direct
aggregate comparisons.

My audit follows the same principle of separating descriptive evidence
from stronger predictive claims.

For my own model, I therefore report the original random split but
treat the client-grouped split as the more conservative validation
result.

The goal is not to maximize the reported score. The goal is to make the
decision-support claim proportional to the evidence.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.